##### Ideal QSVM (SVC Precomputed Kernel) - Spambase

In [1]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

Qiskit: 1.4.4
Aer: 0.17.2
QML: 0.8.4


In [2]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [3]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score
# from imblearn.over_sampling import RandomOverSampler  # For optional balancing

In [4]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.primitives import StatevectorSampler as Sampler
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC, PegasosQSVC

In [5]:
# --- Import Spambase Column Names ---
spambase_columns = [
    "word_freq_make",
    "word_freq_address",
    "word_freq_all",
    "word_freq_3d",
    "word_freq_our",
    "word_freq_over",
    "word_freq_remove",
    "word_freq_internet",
    "word_freq_order",
    "word_freq_mail",
    "word_freq_receive",
    "word_freq_will",
    "word_freq_people",
    "word_freq_report",
    "word_freq_addresses",
    "word_freq_free",
    "word_freq_business",
    "word_freq_email",
    "word_freq_you",
    "word_freq_credit",
    "word_freq_your",
    "word_freq_font",
    "word_freq_000",
    "word_freq_money",
    "word_freq_hp",
    "word_freq_hpl",
    "word_freq_george",
    "word_freq_650",
    "word_freq_lab",
    "word_freq_labs",
    "word_freq_telnet",
    "word_freq_857",
    "word_freq_data",
    "word_freq_415",
    "word_freq_85",
    "word_freq_technology",
    "word_freq_1999",
    "word_freq_parts",
    "word_freq_pm",
    "word_freq_direct",
    "word_freq_cs",
    "word_freq_meeting",
    "word_freq_original",
    "word_freq_project",
    "word_freq_re",
    "word_freq_edu",
    "word_freq_table",
    "word_freq_conference",
    "char_freq_;",
    "char_freq_(",
    "char_freq_[",
    "char_freq_!",
    "char_freq_$",
    "char_freq_#",
    "capital_run_length_average",
    "capital_run_length_longest",
    "capital_run_length_total",
    # finally the target label column:
    "label"
]

# --- 1. Load the Spambase Dataset ---
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\spambase\spambase.data'
df = pd.read_csv(file_path, header=None, names=spambase_columns)
df.drop_duplicates(inplace=True)

In [6]:
# 2. Some basic processing
print(f"Original shape of Spambase data: {df.shape}") # Prints original dataset shape
df.drop_duplicates(inplace=True) # Remove duplicates
print(f"Shape after dropping duplicates: {df.shape}\n") # Then print again the new shape

Original shape of Spambase data: (4210, 58)
Shape after dropping duplicates: (4210, 58)



In [7]:
# Data Preparation

# 1. Split features and target
X = df.drop('label', axis=1)
y = df['label']

# ============================================
# SUBSET DATA (300 samples)
# ============================================
print("=" * 70)
print("Creating 300-sample subset")
print("=" * 70)

# First sample 429 samples from full dataset
X_subset, _, y_subset, _ = train_test_split(
    X, y,
    train_size=429,
    stratify=y,
    random_state=42
)

# Then do 70:30 split on this subset
X_train, X_test, y_train, y_test = train_test_split(
    X_subset, y_subset,
    test_size=0.30,
    random_state=42,
    stratify=y_subset
)

print(f"Subset Training set: {X_train.shape[0]} samples")
print(f"Subset Test set: {X_test.shape[0]} samples\n")

# ============================================
# FULL DATA
# ============================================
print("=" * 70)
print("Creating full dataset split (for scalability demonstration)")
print("=" * 70)

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Full Training set: {X_train_full.shape[0]} samples")
print(f"Full Test set: {X_test_full.shape[0]} samples\n")


Creating 300-sample subset
Subset Training set: 300 samples
Subset Test set: 129 samples

Creating full dataset split (for scalability demonstration)
Full Training set: 2947 samples
Full Test set: 1263 samples



In [8]:
# --- Variance Filtering & Scaling ---
# Variance filtering for SUBSET
selector_variance = VarianceThreshold(threshold=0)
X_train_filtered = selector_variance.fit_transform(X_train)
X_test_filtered = selector_variance.transform(X_test)
remaining_cols = X_train.columns[selector_variance.get_support()]

# Scaling for SUBSET
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filtered)
X_test_scaled = scaler.transform(X_test_filtered)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=remaining_cols)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=remaining_cols)

# Variance filtering for FULL DATASET (use same columns)
X_train_full_filtered = X_train_full[remaining_cols].values
X_test_full_filtered = X_test_full[remaining_cols].values

# Scaling for FULL DATASET
scaler_full = StandardScaler()
X_train_full_scaled = scaler_full.fit_transform(X_train_full_filtered)
X_test_full_scaled = scaler_full.transform(X_test_full_filtered)

X_train_full_scaled_df = pd.DataFrame(X_train_full_scaled, columns=remaining_cols)
X_test_full_scaled_df = pd.DataFrame(X_test_full_scaled, columns=remaining_cols)

In [9]:
# --- Feature Correlation Analysis ---
print("=" * 70)
print("Feature Correlation Analysis")
print("=" * 70)

THRESH = 0.9

# Calculate correlation matrix
corr_matrix_train = X_train_scaled_df.corr().abs()

# Get the upper triangle of the correlation matrix
upper_triangle = corr_matrix_train.where(np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool))

# Find features with correlation greater than the threshold
columns_to_drop = set()
for column in upper_triangle.columns:
    high_corr_partners = upper_triangle.index[upper_triangle[column] > THRESH].tolist()
    if high_corr_partners:
        for partner in high_corr_partners:
            # Check correlation with the TRAINING target variable
            corr_main_vs_target = y_train.corr(X_train_scaled_df[column])
            corr_partner_vs_target = y_train.corr(X_train_scaled_df[partner])
            
            print(f"Found pair: ('{column}', '{partner}') with correlation > {THRESH}")
            if abs(corr_main_vs_target) < abs(corr_partner_vs_target):
                columns_to_drop.add(column)
                print(f"-> Dropping '{column}' (weaker correlation with target)")
            else:
                columns_to_drop.add(partner)
                print(f"-> Dropping '{partner}' (weaker correlation with target)")

to_drop_final = sorted(list(columns_to_drop))
print(f"\nTotal features to drop ({len(to_drop_final)}): {to_drop_final}")

# Drop the identified columns from SUBSET
X_train_selected = X_train_scaled_df.drop(columns=to_drop_final)
X_test_selected = X_test_scaled_df.drop(columns=to_drop_final)

# Drop the same columns from FULL DATASET
X_train_full_selected = X_train_full_scaled_df.drop(columns=to_drop_final)
X_test_full_selected = X_test_full_scaled_df.drop(columns=to_drop_final)

print(f"\nOriginal number of features: {X_train.shape[1]}")
print(f"Number of features after selection: {X_train_selected.shape[1]}\n")


Feature Correlation Analysis
Found pair: ('word_freq_415', 'word_freq_857') with correlation > 0.9
-> Dropping 'word_freq_857' (weaker correlation with target)

Total features to drop (1): ['word_freq_857']

Original number of features: 57
Number of features after selection: 56



c:\Users\User\anaconda3\envs\qsvm_conda\Lib\site-packages\numpy\lib\_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\User\anaconda3\envs\qsvm_conda\Lib\site-packages\numpy\lib\_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [10]:
# --- SelectKBest Feature Selection ---
print("=" * 70)
print("Applying SelectKBest")
print("=" * 70)

k_features = 4  # also number of qubits

# SelectKBest for SUBSET
selector = SelectKBest(score_func=f_classif, k=k_features) 
X_train_kbest = selector.fit_transform(X_train_selected, y_train)
X_test_kbest = selector.transform(X_test_selected)

# Get selected feature names
selected_features = X_train_selected.columns[selector.get_support()].tolist()
print(f"\nSelectKBest: Selected {k_features} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

print(f"\nSubset - Shape after SelectKBest (Train): {X_train_kbest.shape}")
print(f"Subset - Shape after SelectKBest (Test): {X_test_kbest.shape}")

# SelectKBest for FULL DATASET
selector_full = SelectKBest(score_func=f_classif, k=k_features)
X_train_full_kbest = selector_full.fit_transform(X_train_full_selected, y_train_full)
X_test_full_kbest = selector_full.transform(X_test_full_selected)

# Get selected feature names for full dataset
selected_features_full = X_train_full_selected.columns[selector_full.get_support()].tolist()
print(f"\nFull Dataset - Selected {k_features} features:")
for i, feat in enumerate(selected_features_full, 1):
    print(f"  {i}. {feat}")

print(f"\nFull - Shape after SelectKBest (Train): {X_train_full_kbest.shape}")
print(f"Full - Shape after SelectKBest (Test): {X_test_full_kbest.shape}\n")


Applying SelectKBest

SelectKBest: Selected 4 features:
  1. word_freq_over
  2. word_freq_remove
  3. word_freq_your
  4. char_freq_!

Subset - Shape after SelectKBest (Train): (300, 4)
Subset - Shape after SelectKBest (Test): (129, 4)

Full Dataset - Selected 4 features:
  1. word_freq_remove
  2. word_freq_your
  3. word_freq_000
  4. char_freq_$

Full - Shape after SelectKBest (Train): (2947, 4)
Full - Shape after SelectKBest (Test): (1263, 4)



##### Quantum Kernel Implementation

In [11]:
print("=" * 70)
print("Setting up Quantum Kernel (Ideal - Statevector)")
print("=" * 70)

# Feature map
fm = ZZFeatureMap(feature_dimension=k_features, reps=1, entanglement='linear')

# Ideal sampler (statevector - no noise)
sampler = Sampler(default_shots=256)

# Fidelity and kernel
fidelity = ComputeUncompute(sampler=sampler)
qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=fm)

print("Quantum kernel ready (ZZFeatureMap, reps=1, 256 shots)\n")

Setting up Quantum Kernel (Ideal - Statevector)
Quantum kernel ready (ZZFeatureMap, reps=1, 256 shots)



##### Subset Dataset Implementation

In [12]:
# ===================================================================
# Ideal QSVM on Subset
# ===================================================================

print("=" * 70)
print("IDEAL QSVM ON SUBSET (300 train samples)")
print("=" * 70)

# --- Compute Subset Kernel Matrices ---
print("\nComputing kernel matrices for subset...")
start_kernel_subset = time.time()

matrix_train_subset = qkernel.evaluate(x_vec=X_train_kbest)
matrix_test_subset = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)

kernel_time_subset = time.time() - start_kernel_subset
print(f"Subset kernel matrices computed in {kernel_time_subset:.2f} seconds.")

IDEAL QSVM ON SUBSET (300 train samples)

Computing kernel matrices for subset...
Subset kernel matrices computed in 609.06 seconds.


In [13]:
# Save Kernel Matrices
print("\nSaving subset kernel matrices...")
np.save('matrix_train_subset_ideal.npy', matrix_train_subset)
np.save('matrix_test_subset_ideal.npy', matrix_test_subset)
print(f"Saved: matrix_train_subset_ideal.npy ({matrix_train_subset.shape})")
print(f"Saved: matrix_test_subset_ideal.npy ({matrix_test_subset.shape})")


Saving subset kernel matrices...
Saved: matrix_train_subset_ideal.npy ((300, 300))
Saved: matrix_test_subset_ideal.npy ((129, 300))


In [14]:
# --- Grid Search on Subset ---
print("\nStarting Grid Search (Subset)...")

# param_grid = {'C': [0.1, 1, 10, 100]}
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search_subset = GridSearchCV(
    SVC(kernel='precomputed', class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

start_time_subset = time.time()
grid_search_subset.fit(matrix_train_subset, y_train)
qsvc_model_subset = grid_search_subset.best_estimator_
train_time_subset = time.time() - start_time_subset

print(f"Best C parameter: {grid_search_subset.best_params_['C']}")


Starting Grid Search (Subset)...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best C parameter: 10


In [15]:
# --- Evaluation (Subset) ---
y_train_pred_subset = qsvc_model_subset.predict(matrix_train_subset)
y_test_pred_subset = qsvc_model_subset.predict(matrix_test_subset)

train_accuracy_subset = accuracy_score(y_train, y_train_pred_subset)
test_accuracy_subset = accuracy_score(y_test, y_test_pred_subset)
recall_spam_subset = recall_score(y_test, y_test_pred_subset, pos_label=1)
generalization_gap_subset = abs(train_accuracy_subset - test_accuracy_subset)

print("\n" + "=" * 70)
print("Ideal QSVM on Subset (Baseline)")
print("=" * 70)
print(f"Samples Used: {X_train_kbest.shape[0]} train, {X_test_kbest.shape[0]} test")
print(f"Training Accuracy: {train_accuracy_subset:.4f}")
print(f"Test Accuracy: {test_accuracy_subset:.4f}")
print(f"Spam Recall (Class 1): {recall_spam_subset:.4f}")
print(f"Generalization Gap: {generalization_gap_subset:.4f}")
print(f"Training Time: {train_time_subset:.2f} seconds")
print(f"Kernel Computation Time: {kernel_time_subset:.2f} seconds")

print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_test_pred_subset, zero_division=0))


Ideal QSVM on Subset (Baseline)
Samples Used: 300 train, 129 test
Training Accuracy: 0.9533
Test Accuracy: 0.5969
Spam Recall (Class 1): 0.5490
Generalization Gap: 0.3564
Training Time: 4.75 seconds
Kernel Computation Time: 609.06 seconds

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.68      0.63      0.65        78
           1       0.49      0.55      0.52        51

    accuracy                           0.60       129
   macro avg       0.59      0.59      0.59       129
weighted avg       0.61      0.60      0.60       129



##### Full Spambase Dataset

In [16]:
print("=" * 70)
print("IDEAL QSVM ON FULL DATASET")
print("=" * 70)

# --- Compute Full Kernel Matrices ---
print("\nComputing kernel matrices for full dataset (this may take a while)...")
start_kernel_full = time.time()

matrix_train_full = qkernel.evaluate(x_vec=X_train_full_kbest)
matrix_test_full = qkernel.evaluate(x_vec=X_test_full_kbest, y_vec=X_train_full_kbest)

kernel_time_full = time.time() - start_kernel_full
print(f"Full kernel matrices computed in {kernel_time_full:.2f} seconds.")

IDEAL QSVM ON FULL DATASET

Computing kernel matrices for full dataset (this may take a while)...
Full kernel matrices computed in 28384.55 seconds.


In [17]:
# Saving Kernel Matrices
print("\nSaving full kernel matrices...")
np.save('matrix_train_full_ideal.npy', matrix_train_full)
np.save('matrix_test_full_ideal.npy', matrix_test_full)
print(f"Saved: matrix_train_full_ideal.npy ({matrix_train_full.shape})")
print(f"Saved: matrix_test_full_ideal.npy ({matrix_test_full.shape})")


Saving full kernel matrices...


Saved: matrix_train_full_ideal.npy ((2947, 2947))
Saved: matrix_test_full_ideal.npy ((1263, 2947))


In [18]:
# Temporary fix - updating param_grid for full dataset
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

In [19]:
# --- Grid Search on Full Dataset ---
print("\nStarting Grid Search (Full Dataset)...")
grid_search_full = GridSearchCV(
    SVC(kernel='precomputed', class_weight='balanced'),
    param_grid,
    cv=cv,
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

start_time_full = time.time()
grid_search_full.fit(matrix_train_full, y_train_full)
qsvc_model_full = grid_search_full.best_estimator_
train_time_full = time.time() - start_time_full

print(f"Best C parameter: {grid_search_full.best_params_['C']}")


Starting Grid Search (Full Dataset)...
Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best C parameter: 1


In [20]:
# --- Evaluation (Full) ---
y_train_pred_full = qsvc_model_full.predict(matrix_train_full)
y_test_pred_full = qsvc_model_full.predict(matrix_test_full)

train_accuracy_full = accuracy_score(y_train_full, y_train_pred_full)
test_accuracy_full = accuracy_score(y_test_full, y_test_pred_full)
recall_spam_full = recall_score(y_test_full, y_test_pred_full, pos_label=1)
generalization_gap_full = abs(train_accuracy_full - test_accuracy_full)

print("\n" + "=" * 70)
print("FULL DATASET RESULTS: Ideal QSVM")
print("=" * 70)
print(f"Samples Used: {X_train_full_kbest.shape[0]} train, {X_test_full_kbest.shape[0]} test")
print(f"Training Accuracy: {train_accuracy_full:.4f}")
print(f"Test Accuracy: {test_accuracy_full:.4f}")
print(f"Spam Recall (Class 1): {recall_spam_full:.4f}")
print(f"Generalization Gap: {generalization_gap_full:.4f}")
print(f"Training Time: {train_time_full:.2f} seconds")
print(f"Kernel Computation Time: {kernel_time_full:.2f} seconds")

print("\nClassification Report (Test Set):")
print(classification_report(y_test_full, y_test_pred_full, zero_division=0))


FULL DATASET RESULTS: Ideal QSVM
Samples Used: 2947 train, 1263 test
Training Accuracy: 0.8575
Test Accuracy: 0.8211
Spam Recall (Class 1): 0.7560
Generalization Gap: 0.0364
Training Time: 7.11 seconds
Kernel Computation Time: 28384.55 seconds

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       0.84      0.86      0.85       759
           1       0.79      0.76      0.77       504

    accuracy                           0.82      1263
   macro avg       0.81      0.81      0.81      1263
weighted avg       0.82      0.82      0.82      1263



##### Model Evaluation

In [21]:
print("=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"\n{'Metric':<30} {'Subset (300)':<20} {'Full (2947)':<20}")
print("-" * 70)
print(f"{'Training Accuracy':<30} {train_accuracy_subset:<20.4f} {train_accuracy_full:<20.4f}")
print(f"{'Test Accuracy':<30} {test_accuracy_subset:<20.4f} {test_accuracy_full:<20.4f}")
print(f"{'Spam Recall':<30} {recall_spam_subset:<20.4f} {recall_spam_full:<20.4f}")
print(f"{'Generalization Gap':<30} {generalization_gap_subset:<20.4f} {generalization_gap_full:<20.4f}")
print(f"{'Training Time (s)':<30} {train_time_subset:<20.2f} {train_time_full:<20.2f}")
print(f"{'Kernel Time (s)':<30} {kernel_time_subset:<20.2f} {kernel_time_full:<20.2f}")
print("=" * 70)


EXPERIMENT SUMMARY

Metric                         Subset (300)         Full (2947)         
----------------------------------------------------------------------
Training Accuracy              0.9533               0.8575              
Test Accuracy                  0.5969               0.8211              
Spam Recall                    0.5490               0.7560              
Generalization Gap             0.3564               0.0364              
Training Time (s)              4.75                 7.11                
Kernel Time (s)                609.06               28384.55            
